# Chapter 7: What You Have Learnt - Data Processing

## Data Parsing Functions

The following code cell defines a set of Python functions to process a text file containing structured information, specifically bullet points, and convert it into a JSON format. This script is designed to extract key details, including formulas, and normalize the text for consistent processing.

It includes the following helper functions:
*   `normalize_text`: Cleans up the input text by fixing broken header lines (e.g., 'What \n you have \n learnt' to 'What you have learnt') and standardizing line breaks.
*   `split_bullets`: Divides the text into a main header and a list of bullet points.
*   `extract_formulas`: Uses a regular expression to identify and extract potential physics-like equations from a given string.
*   `parse_points`: Iterates through the extracted bullet points, further processes them, and identifies any embedded formulas.

The `parse_file` function orchestrates these steps to read a text file, apply the parsing logic, and return a structured dictionary.

In [1]:
import re
import json


def normalize_text(text):
    # Fix broken header lines like:
    # What \n you have \n learnt → What you have learnt
    text = re.sub(r"What\s*\nyou\s*\nhave\s*\nlearnt", "What you have learnt", text, flags=re.IGNORECASE)

    # Normalize line breaks
    text = re.sub(r"\r", "", text)
    text = re.sub(r"\n+", "\n", text)

    return text.strip()


def split_bullets(text):
    # Split on bullet symbol
    parts = re.split(r"\n?\s*•\s*", text)

    # First part may contain header
    header = parts[0].strip()
    bullets = parts[1:]

    return header, bullets


def extract_formulas(text):
    # Simple heuristic: detect physics equations
    formulas = re.findall(r"[a-zA-Z]+\s*=\s*[^,\n]+", text)
    return formulas if formulas else None


def parse_points(bullets):
    parsed = []

    for bullet in bullets:
        bullet = bullet.strip()

        if not bullet:
            continue

        formulas = extract_formulas(bullet)

        point = {
            "text": bullet
        }

        if formulas:
            point["formulas"] = formulas

        parsed.append(point)

    return parsed


def parse_file(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        raw_text = f.read()

    text = normalize_text(raw_text)
    header, bullets = split_bullets(text)
    points = parse_points(bullets)

    return {
        "section": header,
        "points": points
    }


if __name__ == "__main__":
    input_file = "ch7_whathaveyoulearnt.txt"
    output_file = "whathaveyoulearnt.json"

    data = parse_file(input_file)

    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)

    print(f"Parsed {len(data['points'])} points → {output_file}")

Parsed 7 points → whathaveyoulearnt.json


## Output: `whathaveyoulearnt.json`

The preceding code cell has been executed, processing the input file `ch7_whathaveyoulearnt.txt` and generating `whathaveyoulearnt.json`.

Key improvements and information extracted include:
*   **Header Normalization**: The header "What \n you have \n learnt" was correctly normalized to "What you have learnt".
*   **Standardized Line Breaks**: All line breaks have been unified for consistency.
*   **Formula Extraction**: Physics-like formulas, such as `v = u + at` and `s = ut + ½ at2`, were successfully identified and extracted from the bullet points, appearing as a separate field in the JSON output.

The generated JSON file now provides a structured representation of the "What you have learnt" section, making it easier to programmatically access and analyze its content.